## ReActV2

`ReActV2` é a nova implementação experimental do agente ReAct no DSPy.

Assim como o `ReAct`, ele segue o princípio:

**Reasoning + Acting**

O agente:

1. recebe uma solicitação;
2. analisa o problema;
3. escolhe uma ou mais ferramentas;
4. executa as ferramentas;
5. observa os resultados;
6. continua o processo quando necessário;
7. envia a resposta final.

Entretanto, o `ReActV2` altera a forma como esse processo é representado
internamente.

### ReAct tradicional

O `ReAct` armazena suas etapas em:

    resultado.trajectory

A trajetória possui campos como:

    thought_0
    tool_name_0
    tool_args_0
    observation_0

### ReActV2

O `ReActV2` utiliza:

    resultado.history

O histórico é representado por um objeto:

    dspy.History

Além disso, o resultado possui:

    resultado.termination_reason

que informa por que a execução do agente terminou.

Outra diferença importante é que o `ReActV2` pode solicitar múltiplas
ferramentas dentro de um mesmo turno.

In [ ]:
import os
from dotenv import load_dotenv
import dspy
import json
import requests
import yfinance as yf

load_dotenv()

## Setup - Configuração do Modelo

Carregamos as variáveis de ambiente do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [ ]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Configura o modelo padrão utilizado pelo DSPy
dspy.configure(lm=lm)

## Tool - Consulta de Ações com yfinance

A primeira ferramenta permite consultar a última cotação diária disponível
de uma ação.

O parâmetro `ticker` representa o código utilizado pelo Yahoo Finance.

Exemplos:

- `AAPL` → Apple
- `MSFT` → Microsoft
- `NVDA` → NVIDIA
- `PETR4.SA` → Petrobras
- `VALE3.SA` → Vale

Para ações brasileiras negociadas na B3, normalmente utilizamos o sufixo `.SA`.

A função também calcula a variação percentual entre os dois últimos
fechamentos disponíveis.

In [ ]:
def consultar_acao(ticker: str) -> str:
    """
    Consulta a última cotação diária disponível de uma ação ou ativo
    utilizando yfinance.

    Args:
        ticker: Símbolo do ativo no Yahoo Finance.
                Exemplos:
                AAPL
                MSFT
                NVDA
                PETR4.SA
                VALE3.SA

    Returns:
        String JSON contendo:
        - ticker
        - data da última cotação
        - último fechamento
        - fechamento anterior
        - variação percentual
    """

    try:
        # Remove espaços e padroniza o ticker em letras maiúsculas
        ticker = ticker.strip().upper()

        # Cria um objeto representando o ativo no Yahoo Finance
        ativo = yf.Ticker(ticker)

        # Obtém os últimos dias de negociação
        historico = ativo.history(
            period="5d",       # Últimos 5 dias
            interval="1d",     # Dados diários
            auto_adjust=False  # Mantém preços sem ajuste automático
        )

        # Remove linhas nas quais o fechamento não está disponível
        historico = historico.dropna(subset=["Close"])

        # Verifica se algum dado foi encontrado
        if historico.empty:
            return json.dumps(
                {
                    "erro": f"Nenhum dado encontrado para o ticker {ticker}."
                },
                ensure_ascii=False
            )

        # Último preço de fechamento disponível
        preco_atual = float(historico["Close"].iloc[-1])

        # Data correspondente ao último fechamento
        data_atual = historico.index[-1].strftime("%Y-%m-%d")

        # Calcula a variação em relação ao fechamento anterior
        if len(historico) >= 2:

            preco_anterior = float(
                historico["Close"].iloc[-2]
            )

            variacao = (
                (preco_atual - preco_anterior)
                / preco_anterior
            ) * 100

        else:
            preco_anterior = None
            variacao = None

        # Estrutura a resposta que será devolvida ao agente
        resultado = {
            "ticker": ticker,
            "data": data_atual,
            "ultimo_fechamento": round(preco_atual, 2),
            "fechamento_anterior": (
                round(preco_anterior, 2)
                if preco_anterior is not None
                else None
            ),
            "variacao_percentual": (
                round(variacao, 2)
                if variacao is not None
                else None
            )
        }

        # ReAct recebe uma representação textual do resultado
        return json.dumps(
            resultado,
            ensure_ascii=False
        )

    except Exception as erro:

        return json.dumps(
            {
                "erro": f"Falha ao consultar {ticker}: {str(erro)}"
            },
            ensure_ascii=False
        )

## Testando a Tool do Yahoo Finance

Antes de fornecer a ferramenta ao agente ReAct, podemos executá-la
diretamente para verificar seu funcionamento.

In [ ]:
consultar_acao("NVDA")

In [ ]:
consultar_acao("PETR4.SA")

## Tool - Chuck Norris API

A segunda ferramenta consulta a API pública `chucknorris.io`.

A API oferece piadas/fatos satíricos de Chuck Norris.

É possível:

- obter uma piada completamente aleatória;
- selecionar uma categoria específica.

Quando nenhuma categoria é informada, utilizamos:

    /jokes/random

Quando uma categoria é informada:

    /jokes/random?category=categoria

Antes de realizar a consulta por categoria, nossa função também verifica
quais categorias são válidas utilizando:

    /jokes/categories

In [ ]:
def obter_piada_chuck_norris(categoria: str = "") -> str:
    """
    Obtém uma piada/fato satírico de Chuck Norris utilizando
    a API pública chucknorris.io.

    Args:
        categoria: Categoria opcional da piada.

                   Exemplos comuns:
                   dev
                   movie
                   food
                   science

                   Se for uma string vazia, uma piada aleatória
                   será retornada.

    Returns:
        Texto da piada retornada pela API.
    """

    try:

        # Endpoint utilizado para obter uma piada
        url = "https://api.chucknorris.io/jokes/random"

        # Parâmetros enviados na requisição
        params = {}

        # Normaliza a categoria informada
        categoria = categoria.strip().lower()

        # Se uma categoria foi solicitada
        if categoria:

            # Consulta primeiro as categorias disponíveis na API
            categorias_response = requests.get(
                "https://api.chucknorris.io/jokes/categories",
                timeout=10
            )

            # Gera exceção caso a requisição HTTP tenha falhado
            categorias_response.raise_for_status()

            # Converte a resposta JSON para uma lista Python
            categorias = categorias_response.json()

            # Verifica se a categoria solicitada existe
            if categoria not in categorias:

                return (
                    f"Categoria '{categoria}' não encontrada. "
                    f"Categorias disponíveis: {', '.join(categorias)}"
                )

            # Adiciona a categoria à chamada da API
            params["category"] = categoria

        # Realiza a requisição para obter a piada
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        # Gera exceção caso a API retorne um erro HTTP
        response.raise_for_status()

        # Converte o JSON retornado pela API
        dados = response.json()

        # O campo "value" contém o texto da piada
        return dados["value"]

    except requests.RequestException as erro:

        return (
            "Erro ao acessar a API Chuck Norris: "
            f"{str(erro)}"
        )

## Testando a Tool Chuck Norris

Podemos verificar a ferramenta diretamente antes de disponibilizá-la
ao agente.

In [ ]:
obter_piada_chuck_norris()

In [ ]:
obter_piada_chuck_norris("dev")

## Criando a Signature

A `Signature` define a interface entre nossa aplicação e o agente ReAct.

Teremos:

### Entrada

`pergunta`

Representa a solicitação feita pelo usuário.

### Saída

`resposta`

Representa a resposta final produzida pelo agente após utilizar,
quando necessário, uma ou mais ferramentas.

In [ ]:
class AssistenteComFerramentas(dspy.Signature):
    """
    Responda à pergunta do usuário.

    Use as ferramentas disponíveis sempre que forem necessárias para
    obter dados financeiros ou uma piada/fato de Chuck Norris.
    """

    # Pergunta recebida pelo agente
    pergunta: str = dspy.InputField(
        desc="Pergunta ou solicitação feita pelo usuário"
    )

    # Resposta produzida depois da execução das ferramentas necessárias
    resposta: str = dspy.OutputField(
        desc=(
            "Resposta final em português, clara e baseada "
            "nos resultados das ferramentas"
        )
    )

## Criar Módulo ReActV2

Agora criamos o agente utilizando:

    dspy.ReActV2()

Os principais parâmetros são:

- `signature`: define entradas e saídas do agente;
- `tools`: lista de funções que podem ser executadas;
- `max_iters`: número máximo de iterações do agente.

Disponibilizamos duas ferramentas:

1. `consultar_acao`
2. `obter_piada_chuck_norris`

O `ReActV2` também adicionará automaticamente a ferramenta interna:

    submit

Essa ferramenta é utilizada para devolver a resposta final.

In [ ]:
agente = dspy.ReActV2(
    AssistenteComFerramentas,  # Signature utilizada pelo agente

    tools=[
        consultar_acao,              # Tool baseada em yfinance
        obter_piada_chuck_norris     # Tool baseada em chucknorris.io
    ],

    max_iters=6  # Número máximo de turnos do agente
)

## Executar ReActV2 - Consulta Financeira

Primeiro realizamos uma tarefa que precisa apenas da ferramenta
financeira.

O modelo deverá reconhecer que a pergunta necessita de dados externos
e selecionar `consultar_acao`.

In [ ]:
resultado_financeiro = agente(
    pergunta=(
        "Qual foi a última cotação disponível da NVDA "
        "e qual foi sua variação percentual em relação "
        "ao pregão anterior?"
    )
)

resultado_financeiro

## Ver Resposta Final

O campo `resposta` contém a resposta final construída pelo agente
depois de consultar as ferramentas necessárias.

In [ ]:
print("=" * 70)
print("RESPOSTA FINAL")
print("=" * 70)

print(resultado_financeiro.resposta)

## Motivo da Finalização

O `ReActV2` adiciona ao resultado o atributo:

    termination_reason

Esse campo informa por que o loop do agente foi encerrado.

Quando o agente chama corretamente a ferramenta interna `submit`,
normalmente encontramos:

    submit

Se for necessária uma tentativa forçada de finalização, podemos
encontrar:

    forced_submit

Outros valores podem indicar que o agente atingiu alguma condição
de parada antes de produzir a saída normalmente.

In [ ]:
print("=" * 70)
print("MOTIVO DA FINALIZAÇÃO")
print("=" * 70)

print(resultado_financeiro.termination_reason)

## Executar ReAct - Chuck Norris

Agora fazemos uma solicitação que necessita apenas da API Chuck Norris.

O agente deverá identificar que a ferramenta financeira não é necessária
e selecionar `obter_piada_chuck_norris`.

In [ ]:
resultado_piada = agente(
    pergunta=(
        "Conte uma piada de Chuck Norris "
        "relacionada à categoria dev."
    )
)

print(resultado_piada.resposta)

## Executar ReActV2 com Múltiplas Tools

Agora criamos uma tarefa que necessita de duas fontes externas.

O agente deverá:

1. consultar a cotação da NVIDIA;
2. calcular/obter a variação em relação ao pregão anterior;
3. consultar a API Chuck Norris;
4. produzir uma resposta contendo as duas informações.

Esse exemplo é particularmente interessante para `ReActV2` porque
essa implementação é capaz de representar múltiplas chamadas de
ferramentas dentro de um mesmo turno.

In [ ]:
resultado = agente(
    pergunta=(
        "Consulte a última cotação disponível da NVDA e informe "
        "a variação percentual em relação ao pregão anterior. "
        "Depois conte uma piada de Chuck Norris da categoria dev."
    )
)

resultado

## Ver Resposta Final

Agora exibimos somente a resposta final criada pelo agente após
executar as ferramentas necessárias.

In [ ]:
print("=" * 70)
print("RESPOSTA FINAL")
print("=" * 70)

print(resultado.resposta)

## Inspecionando o History

No `ReAct` tradicional utilizávamos:

    resultado.trajectory

No `ReActV2` utilizamos:

    resultado.history

O objeto retornado é uma instância de:

    dspy.History

O histórico possui uma lista chamada:

    messages

Cada elemento dessa lista representa um evento estruturado ocorrido
durante a execução do agente.

Os eventos podem conter informações como:

- entrada original;
- pensamento do agente;
- chamadas de ferramentas;
- resultados das ferramentas;
- saída final enviada através de `submit`.

In [ ]:
print("=" * 70)
print("HISTORY")
print("=" * 70)

print(resultado.history.messages)

## Visualizando Cada Turno do ReActV2

Como `history.messages` é uma lista, podemos percorrer os eventos
individualmente.

Isso torna mais fácil compreender como o agente evoluiu durante
a execução.

In [ ]:
for numero, mensagem in enumerate(
    resultado.history.messages,
    start=1
):
    print("=" * 70)
    print(f"TURNO {numero}")
    print("=" * 70)

    print(mensagem)

    print()

## Identificando Chamadas de Ferramentas

Os eventos do `ReActV2` podem possuir um campo chamado:

    tool_calls

Esse campo contém um objeto `dspy.ToolCalls`.

Podemos percorrer o histórico e identificar quais turnos envolveram
ferramentas.

In [ ]:
for numero, mensagem in enumerate(
    resultado.history.messages,
    start=1
):

    if "tool_calls" in mensagem:

        print("=" * 70)
        print(f"TOOL CALLS - TURNO {numero}")
        print("=" * 70)

        print(mensagem["tool_calls"])

        print()

## Continuando a partir de um History

O `History` criado pelo `ReActV2` pode ser fornecido novamente ao agente.

Isso permite continuar uma execução utilizando o histórico estruturado
produzido anteriormente.

Utilizamos:

    history=resultado.history

Dessa forma, o agente recebe os eventos anteriores como contexto para
a nova execução.

In [ ]:
resultado_continuacao = agente(
    pergunta=(
        "Agora consulte também a PETR4.SA e compare "
        "a variação dela com a ação consultada anteriormente."
    ),

    # Reutiliza o histórico estruturado da execução anterior
    history=resultado.history
)

print(resultado_continuacao.resposta)

## Native Function Calling

O `ReActV2` foi projetado para trabalhar com chamadas estruturadas
de ferramentas.

O DSPy permite habilitar explicitamente o suporte a native function
calling através do `ChatAdapter`.

Também podemos permitir que o modelo solicite múltiplas ferramentas
independentes dentro do mesmo turno através de:

    parallel_tool_calls=True

É importante observar que esse parâmetro permite ao modelo **solicitar**
múltiplas ferramentas no mesmo turno.

Atualmente, o `ReActV2` ainda executa essas chamadas uma após a outra
no Python.

In [ ]:
adapter = dspy.ChatAdapter(
    use_native_function_calling=True,  # Usa function calling nativo do modelo
    parallel_tool_calls=True           # Permite múltiplas tools em um turno
)

## Executando com Native Function Calling

Utilizamos `dspy.context()` para alterar temporariamente o Adapter
da execução.

Somente o código dentro do bloco utilizará essa configuração.

In [ ]:
with dspy.context(adapter=adapter):

    resultado_native = agente(
        pergunta=(
            "Consulte a última cotação disponível da NVDA "
            "e da PETR4.SA. Informe a variação percentual "
            "de cada uma. Depois conte uma piada de "
            "Chuck Norris da categoria dev."
        )
    )

In [ ]:
print("=" * 70)
print("RESPOSTA FINAL")
print("=" * 70)

print(resultado_native.resposta)


print("\n" + "=" * 70)
print("TERMINATION REASON")
print("=" * 70)

print(resultado_native.termination_reason)

In [ ]:
print("=" * 70)
print("HISTORY")
print("=" * 70)

for numero, mensagem in enumerate(
    resultado_native.history.messages,
    start=1
):

    print(f"\nTURNO {numero}")
    print("-" * 70)

    print(mensagem)

## Inspecionando o Histórico do Modelo

`resultado.history` representa o histórico estruturado mantido pelo
agente ReActV2.

Já:

    dspy.inspect_history()

mostra as chamadas realizadas pelo DSPy ao Language Model.

Portanto são conceitos relacionados, mas diferentes:

- `resultado.history` → estado estruturado do agente;
- `dspy.inspect_history()` → chamadas realizadas ao modelo.

In [ ]:
# Exibe as últimas chamadas realizadas ao Language Model
dspy.inspect_history(n=20)

## Comparação: ReAct vs ReActV2

| Característica | ReAct | ReActV2 |
|---|---|---|
| Histórico | `trajectory` | `dspy.History` |
| Diagnóstico | `prediction.trajectory` | `prediction.history` |
| Motivo da finalização | Não é o mecanismo principal | `termination_reason` |
| Representação das Tools | nome + argumentos | `dspy.ToolCalls` |
| Tools por turno | Uma | Uma ou mais |
| Finalização | `finish` | `submit` |
| Extração final | Nova chamada ao LM | Saída enviada diretamente por `submit` |
| Continuação de histórico | Não projetado para isso | `history=...` |
| Function calling nativo | Modelo antigo de execução | Suporte estruturado |

O ReActV2 mantém cada interação como eventos estruturados.

Isso facilita:

- continuidade do agente;
- inspeção das ferramentas executadas;
- associação entre tool calls e seus resultados;
- uso de native function calling;
- aproveitamento de prompt caching;
- execução de múltiplas chamadas de ferramentas no mesmo turno.